In [ ]:
"""
The code for doing simulation. Running the code will generate results in csv file.

The simulation results used in the paper are put into folder ../data
"""

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
import numpy as np
import pandas as pd
from create_data import *
from compute_MI_singlew import M1M2_w, muvar_w, MI_wijxl
from compute_MI_multiw import sigma_ww, MI_nw_xl
import matplotlib.pyplot as plt

In [ ]:
def generate_random_connections(d):
    """
    Given a dxd neural net, randomly generate d unique connections
    """
    # Generate all possible tuples (i, j) where 1 <= i < j <= d
    all_possible_tuples = [(i, j) for i in range(1, d + 1) for j in range(i + 1, d + 1)]

    # Determine the maximum possible length of the list
    max_length = len(all_possible_tuples)
    bound = 20

    # Randomly select the length of the list within the range 1 to max_length
    list_length = np.random.randint(1, np.min([max_length, bound]) + 1) 

    # Randomly sample from the possible tuples to create the list
    selected_indices = np.random.choice(max_length, size=list_length, replace=False)
    random_tuples = [all_possible_tuples[i] for i in selected_indices]
    return random_tuples


# Initialize an empty list to store the generated samples
dataframe = []

# Number of samples to generate
num_samples = 20000  # Adjust as needed

# Generate samples
for _ in range(num_samples):
    # randomly generate data
    d = 20  # MI doesn't depend on d
    N = np.random.randint(2, 50 + 1)
    range_mu = 1
    range_s = 1.2 # 1.2 for different cov analysis, 1 for the same cov analysis
    # change this line t
    data = create_patterns_general_eigen(N, d, range_mu, range_s) # create_patterns_general_eigen # create_patterns_samecov(N, d, range_mu, range_s)

    # randomly pick n weights
    w_list = generate_random_connections(d)
    n = len(w_list)
    print(_, N, n)

    # Compute mutual information
    MI = MI_nw_xl(data, w_list, np.random.randint(1, N + 1))

    # if MI cannot be computed, abandon it
    if np.isnan(MI) or np.isinf(MI) or MI < 0:
        MI = 'none'
        print('------------------none!')
        continue

    # Store the sample as a dictionary
    sample = {
        'N': N,
        'n': n,
        'MI': MI
    }
    # append to dataframe
    dataframe.append(sample)

# Convert the list of dictionaries to a pandas DataFrame
df = pd.DataFrame(dataframe)

# Display the DataFrame
print(df)

# Save the DataFrame to a CSV file
df.to_csv('generated_samples.csv', index=False)